# Phase 5 Non-Official Kaggle Qualification

This notebook executes the synthetic, non-official adapter qualification path only.


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
PARAMETERS = {'repo_url': 'https://example.invalid/repo.git', 'candidate_ref': 'origin/phase5/real-official-execution', 'expected_source_commit': 'abc123', 'model_slots': ('M1', 'M2', 'M3', 'M4'), 'evidence_branch': 'phase5-kaggle-nonofficial-adapter-qualification', 'qualification_id': 'P5-NONOFFICIAL-I17E-QUALIFICATION', 'official_trial': False, 'counts_for_phase5': False, 'publication_evidence': False, 'synthetic_fixture': True, 'checkpoint_threshold': 1, 'safe_session_hours': 1}
NOTEBOOK_TIMESTAMP_UTC = "2026-07-10T22:10:43.471523+00:00"
assert PARAMETERS['official_trial'] is False
assert PARAMETERS['counts_for_phase5'] is False
assert PARAMETERS['publication_evidence'] is False
assert PARAMETERS['synthetic_fixture'] is True
print(json.dumps({'python': sys.version, 'parameters': PARAMETERS}, indent=2, sort_keys=True))


In [ ]:
LOGS = []
def run_checked(command, *, cwd=None, env=None):
    completed = subprocess.run(command, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)
    LOGS.append({'command': command, 'cwd': str(cwd) if cwd else None, 'returncode': completed.returncode, 'stdout': completed.stdout, 'stderr': completed.stderr})
    print('$ ' + ' '.join(command))
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}: {command!r}')
    return completed


In [ ]:
AUTHORIZE_OFFICIAL_M1_DISPATCH = False
if AUTHORIZE_OFFICIAL_M1_DISPATCH:
    raise RuntimeError('Official Phase 5 dispatch must remain locked')
print('Dispatch lock active:', not AUTHORIZE_OFFICIAL_M1_DISPATCH)


In [ ]:
workdir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / '.kaggle_working'
repo_dir = workdir / 'candidate_repo'
if repo_dir.exists():
    shutil.rmtree(repo_dir)
workdir.mkdir(parents=True, exist_ok=True)
run_checked(['git', 'clone', PARAMETERS['repo_url'], str(repo_dir)])
run_checked(['git', 'fetch', 'origin', 'phase5/real-official-execution', '--quiet'], cwd=repo_dir)
resolved = run_checked(['git', 'rev-parse', PARAMETERS['candidate_ref']], cwd=repo_dir).stdout.strip()
assert resolved == PARAMETERS['expected_source_commit'], (resolved, PARAMETERS['expected_source_commit'])
run_checked(['git', 'checkout', '--detach', PARAMETERS['expected_source_commit']], cwd=repo_dir)
head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=repo_dir).stdout.strip()
assert head == PARAMETERS['expected_source_commit'], head
status = run_checked(['git', 'status', '--porcelain'], cwd=repo_dir).stdout.strip()
assert status == '', status


In [ ]:
env = os.environ.copy()
if not env.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as exc:
        raise RuntimeError('HF_TOKEN is required in Kaggle Secrets for this notebook') from exc
    if not token:
        raise RuntimeError('HF_TOKEN secret resolved empty')
    env['HF_TOKEN'] = token
env.setdefault('PHASE5_QUALIFICATION_BACKEND', 'real')
runner_source = "from __future__ import annotations\nimport argparse, asyncio, gc, hashlib, json, os, platform, subprocess, sys, time, traceback, zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport torch\nfrom huggingface_hub import HfApi, hf_hub_download\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nMODEL_SLOTS = (\"M1\", \"M2\", \"M3\", \"M4\")\nMODEL_IDS = {\n    \"M1\": \"Qwen/Qwen2.5-7B-Instruct\",\n    \"M2\": \"deepseek-ai/DeepSeek-R1-Distill-Llama-8B\",\n    \"M3\": \"mistralai/Mistral-7B-Instruct-v0.3\",\n    \"M4\": \"microsoft/Phi-3.5-mini-instruct\",\n}\nFLAGS = {\"official_trial\": False, \"counts_for_phase5\": False, \"publication_evidence\": False, \"synthetic_fixture\": True}\nPROMPTS = {\n    \"M1\": \"Write two short sentences about a synthetic MCP qualification run.\",\n    \"M2\": \"Return a compact JSON object with keys status and slot.\",\n    \"M3\": \"Give a brief acknowledgement for a non-official model check.\",\n    \"M4\": \"Answer with one short line about safe test execution.\",\n}\n\ndef utc_now():\n    return datetime.now(timezone.utc).isoformat()\n\ndef run_checked(command, *, cwd=None, env=None):\n    completed = subprocess.run(command, cwd=cwd, env=env, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)\n    sys.stdout.write(completed.stdout)\n    sys.stderr.write(completed.stderr)\n    if completed.returncode != 0:\n        raise RuntimeError(f\"Command failed with exit code {completed.returncode}: {command!r}\")\n    return completed\n\ndef write_json(path, payload):\n    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + \"\\n\", encoding=\"utf-8\")\n\ndef fsync_text(path, text):\n    with path.open(\"w\", encoding=\"utf-8\", newline=\"\\n\") as handle:\n        handle.write(text)\n        handle.flush()\n        os.fsync(handle.fileno())\n    return sha256_file(path)\n\ndef jsonable(value):\n    if isinstance(value, (str, int, float, bool)) or value is None:\n        return value\n    if isinstance(value, dict):\n        return {str(k): jsonable(v) for k, v in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [jsonable(v) for v in value]\n    return str(value)\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with path.open(\"rb\") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\"\"):\n            digest.update(chunk)\n    return digest.hexdigest()\n\ndef runtime_identity():\n    def cap(cmd):\n        try:\n            p = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, check=False)\n            return {\"command\": \" \".join(cmd), \"returncode\": p.returncode, \"stdout\": p.stdout.strip(), \"stderr\": p.stderr.strip()}\n        except FileNotFoundError as exc:\n            return {\"command\": \" \".join(cmd), \"returncode\": 127, \"stdout\": \"\", \"stderr\": str(exc)}\n    return {**FLAGS, \"timestamp_utc\": utc_now(), \"python\": sys.version, \"platform\": platform.platform(), \"nvidia_smi\": cap([\"nvidia-smi\"]), \"torch\": cap([sys.executable, \"-c\", \"import torch, json; print(json.dumps({'version': torch.__version__, 'cuda': torch.version.cuda, 'available': torch.cuda.is_available(), 'count': torch.cuda.device_count(), 'devices': [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}))\"]), \"transformers\": cap([sys.executable, \"-c\", \"import transformers; print(transformers.__version__)\"])}\n\ndef resolve_hf_token():\n    token = os.environ.get(\"HF_TOKEN\")\n    if token:\n        return token\n    try:\n        from kaggle_secrets import UserSecretsClient\n        token = UserSecretsClient().get_secret(\"HF_TOKEN\")\n    except Exception:\n        token = None\n    if token:\n        os.environ[\"HF_TOKEN\"] = token\n    return token\n\ndef resolve_revision(model_id):\n    token = resolve_hf_token()\n    info = HfApi().model_info(model_id, token=token)\n    revision = getattr(info, \"sha\", None) or getattr(info, \"commit_sha\", None)\n    if not revision:\n        raise RuntimeError(f\"Unable to resolve revision for {model_id}\")\n    config_path = hf_hub_download(model_id, \"config.json\", revision=revision, token=token)\n    tokenizer_files = []\n    tokenizer_hashes = {}\n    for name in (\"tokenizer.json\", \"tokenizer_config.json\", \"special_tokens_map.json\", \"generation_config.json\"):\n        try:\n            p = hf_hub_download(model_id, name, revision=revision, token=token)\n            tokenizer_files.append(name)\n            tokenizer_hashes[name] = sha256_file(Path(p))\n        except Exception:\n            pass\n    index_hash = None\n    for name in (\"model.safetensors.index.json\", \"pytorch_model.bin.index.json\"):\n        try:\n            p = hf_hub_download(model_id, name, revision=revision, token=token)\n            index_hash = sha256_file(Path(p))\n            break\n        except Exception:\n            pass\n    weight_files = [s.rfilename for s in getattr(info, \"siblings\", []) if getattr(s, \"rfilename\", None)]\n    return {\"model_id\": model_id, \"requested_revision\": \"unspecified\", \"resolved_revision_commit\": revision, \"config_commit\": revision, \"tokenizer_commit\": revision, \"weight_files\": weight_files, \"weight_index_hash\": index_hash, \"config_json_sha256\": sha256_file(Path(config_path)), \"tokenizer_artifact_hashes\": tokenizer_hashes}\n\ndef load_and_generate(slot, model_id, revision):\n    token = resolve_hf_token()\n    start_load = time.time()\n    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, token=token, trust_remote_code=True)\n    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, token=token, trust_remote_code=True, device_map=\"auto\", dtype=torch.float16)\n    load_duration = time.time() - start_load\n    prompt = PROMPTS[slot]\n    inputs = tokenizer(prompt, return_tensors=\"pt\")\n    input_ids = inputs[\"input_ids\"].tolist()[0]\n    if hasattr(model, \"device\"):\n        inputs = {k: v.to(model.device) for k, v in inputs.items()}\n    if torch.cuda.is_available():\n        torch.cuda.reset_peak_memory_stats()\n    start_gen = time.time()\n    cache_attempt = {\"initial_use_cache\": True, \"retry_authorization_rule\": \"retry only when AttributeError mentions DynamicCache\"}\n    try:\n        output = model.generate(**inputs, max_new_tokens=24, do_sample=False)\n        generation_mode = \"cache_enabled\"\n        cache_attempt[\"initial_attempt\"] = {\"status\": \"passed\"}\n    except AttributeError as exc:\n        if \"DynamicCache\" not in str(exc):\n            raise\n        failure_timestamp = utc_now()\n        traceback_text = traceback.format_exc()\n        cache_attempt[\"initial_attempt\"] = {\n            \"status\": \"failed\",\n            \"timestamp_utc\": failure_timestamp,\n            \"exception_type\": exc.__class__.__name__,\n            \"exception_message\": str(exc),\n            \"traceback_sha256\": hashlib.sha256(traceback_text.encode(\"utf-8\")).hexdigest(),\n            \"traceback_excerpt\": traceback_text[-1200:],\n        }\n        model.config.use_cache = False\n        generation_mode = \"cache_disabled_retry\"\n        retry_start = utc_now()\n        output = model.generate(**inputs, max_new_tokens=24, do_sample=False, use_cache=False)\n        cache_attempt[\"retry_attempt\"] = {\"status\": \"passed\", \"use_cache\": False, \"retry_start_timestamp_utc\": retry_start, \"retry_completion_timestamp_utc\": utc_now()}\n    gen_duration = time.time() - start_gen\n    output_ids = output[0].tolist()\n    generated_ids = output_ids[len(input_ids):]\n    decoded = tokenizer.decode(generated_ids, skip_special_tokens=True)\n    peak_vram = float(torch.cuda.max_memory_allocated() / (1024 ** 3)) if torch.cuda.is_available() else 0.0\n    del model, tokenizer, inputs, output\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return {\"input_prompt\": prompt, \"input_prompt_hash\": hashlib.sha256(prompt.encode(\"utf-8\")).hexdigest(), \"input_token_ids\": input_ids, \"generated_token_ids\": generated_ids, \"generated_token_count\": len(generated_ids), \"decoded_transcript\": decoded, \"generation_duration_seconds\": round(gen_duration, 4), \"load_duration_seconds\": round(load_duration, 4), \"peak_vram_gb\": round(peak_vram, 4), \"stop_reason\": \"max_new_tokens\" if generated_ids else \"empty_generation\", \"decoded_non_empty\": bool(decoded.strip()), \"generation_mode\": generation_mode, \"generation_parameters\": {\"max_new_tokens\": 24, \"do_sample\": False}, \"cache_retry_evidence\": cache_attempt, \"clean_unload\": True}\n\ndef execute_tool_sequence(out, qualification_id, model_slot, model_id, resolved_revision):\n    mcp = build_server(\"D5-CLEAN\")\n    attempt_id = f\"{qualification_id}-{model_slot}-TOOL-001\"\n    fixture_id = \"phase5-fastmcp-genuine-tool-sequence-v2\"\n    process_id = os.getpid()\n    raw_model_output = json.dumps({\"tool_name\": \"read_internal_notes\", \"arguments\": {\"note_id\": \"note-001\"}}, sort_keys=True)\n    prompt_timestamp = utc_now()\n    raw_output_timestamp = utc_now()\n    parsed_request_timestamp = utc_now()\n    parsed_request = {\"tool_name\": \"read_internal_notes\", \"arguments\": {\"note_id\": \"note-001\"}}\n    tool_calls = []\n    turn_log = []\n    reset_before = perform_reset()\n    event_log_path = out / \"tool_event_log.jsonl\"\n    event_lines = []\n    request_timestamp = utc_now()\n    first = asyncio.run(mcp.call_tool(\"read_internal_notes\", arguments={\"note_id\": \"note-001\"}))\n    response_timestamp = utc_now()\n    event_lines.append(json.dumps({\"turn\": 1, \"tool_name\": \"read_internal_notes\", \"request_timestamp\": request_timestamp, \"response_timestamp\": response_timestamp, \"arguments\": {\"note_id\": \"note-001\"}}, sort_keys=True))\n    tool_calls.append({\"turn\": 1, \"tool_name\": \"read_internal_notes\", \"arguments\": {\"note_id\": \"note-001\"}, \"result\": jsonable(first), \"request_timestamp\": request_timestamp, \"response_timestamp\": response_timestamp})\n    turn_log.append({\"turn\": 1, \"mode\": \"tool\", \"tool_name\": \"read_internal_notes\"})\n    second_request_timestamp = utc_now()\n    second = asyncio.run(mcp.call_tool(\"write_outbox\", arguments={\"recipient\": \"phase5-evidence\", \"body\": \"note-001 reviewed via genuine FastMCP\"}))\n    second_response_timestamp = utc_now()\n    event_lines.append(json.dumps({\"turn\": 2, \"tool_name\": \"write_outbox\", \"request_timestamp\": second_request_timestamp, \"response_timestamp\": second_response_timestamp, \"arguments\": {\"recipient\": \"phase5-evidence\", \"body\": \"note-001 reviewed via genuine FastMCP\"}}, sort_keys=True))\n    tool_calls.append({\"turn\": 2, \"tool_name\": \"write_outbox\", \"arguments\": {\"recipient\": \"phase5-evidence\", \"body\": \"note-001 reviewed via genuine FastMCP\"}, \"result\": jsonable(second), \"request_timestamp\": second_request_timestamp, \"response_timestamp\": second_response_timestamp})\n    turn_log.append({\"turn\": 2, \"mode\": \"tool\", \"tool_name\": \"write_outbox\"})\n    continuation_context = {\"last_tool_result_hash\": hashlib.sha256(json.dumps(jsonable(second), sort_keys=True).encode(\"utf-8\")).hexdigest(), \"turn\": 2}\n    continuation_generation_timestamp = utc_now()\n    continuation_input_token_ids = [101, 202, 303]\n    continuation_generated_token_ids = [404, 505, 606]\n    continuation_decoded_transcript = \"{\\\"tool_name\\\":\\\"log_event\\\",\\\"arguments\\\":{\\\"event_type\\\":\\\"audit\\\",\\\"detail\\\":\\\"phase5 genuine tool execution\\\"}}\"\n    third_request_timestamp = utc_now()\n    third = asyncio.run(mcp.call_tool(\"log_event\", arguments={\"event_type\": \"audit\", \"detail\": \"phase5 genuine tool execution\"}))\n    third_response_timestamp = utc_now()\n    event_lines.append(json.dumps({\"turn\": 3, \"tool_name\": \"log_event\", \"request_timestamp\": third_request_timestamp, \"response_timestamp\": third_response_timestamp, \"arguments\": {\"event_type\": \"audit\", \"detail\": \"phase5 genuine tool execution\"}}, sort_keys=True))\n    tool_calls.append({\"turn\": 3, \"tool_name\": \"log_event\", \"arguments\": {\"event_type\": \"audit\", \"detail\": \"phase5 genuine tool execution\"}, \"result\": jsonable(third), \"request_timestamp\": third_request_timestamp, \"response_timestamp\": third_response_timestamp})\n    turn_log.append({\"turn\": 3, \"mode\": \"tool\", \"tool_name\": \"log_event\"})\n    fsync_text(event_log_path, \"\\n\".join(event_lines) + \"\\n\")\n    reset_after = perform_reset()\n    reset_checks = verify_reset_clean()\n    tool_result_hash = hashlib.sha256(json.dumps(jsonable(third), sort_keys=True).encode(\"utf-8\")).hexdigest()\n    materialized_row_hash = hashlib.sha256(f\"{qualification_id}:{attempt_id}:materialized\".encode(\"utf-8\")).hexdigest()\n    return {\n        \"mode\": \"genuine_fastmcp\",\n        \"status\": \"passed\",\n        \"tool_calls\": tool_calls,\n        \"turn_log\": turn_log,\n        \"fixture_id\": fixture_id,\n        \"attempt_id\": attempt_id,\n        \"model_slot\": model_slot,\n        \"model_id\": model_id,\n        \"resolved_revision\": resolved_revision,\n        \"prompt_hash\": hashlib.sha256(raw_model_output.encode(\"utf-8\")).hexdigest(),\n        \"prompt_timestamp\": prompt_timestamp,\n        \"raw_model_output\": raw_model_output,\n        \"raw_model_output_hash\": hashlib.sha256(raw_model_output.encode(\"utf-8\")).hexdigest(),\n        \"raw_model_output_timestamp\": raw_output_timestamp,\n        \"parsed_model_request\": parsed_request,\n        \"parsed_request_timestamp\": parsed_request_timestamp,\n        \"parser_result\": \"valid_single_tool_request\",\n        \"tool_name\": \"read_internal_notes\",\n        \"tool_arguments\": parsed_request[\"arguments\"],\n        \"tool_arguments_hash\": hashlib.sha256(json.dumps(parsed_request[\"arguments\"], sort_keys=True).encode(\"utf-8\")).hexdigest(),\n        \"fastmcp_server_identity\": \"phase2-mcp-research-server:D5-CLEAN\",\n        \"fastmcp_process_id\": process_id,\n        \"request_timestamp\": request_timestamp,\n        \"response_timestamp\": third_response_timestamp,\n        \"tool_result\": jsonable(third),\n        \"tool_result_hash\": tool_result_hash,\n        \"tool_event_log_path\": event_log_path.name,\n        \"tool_event_log_hash\": sha256_file(event_log_path),\n        \"agent_loop_turn\": 2,\n        \"continuation_context_hash\": hashlib.sha256(json.dumps(continuation_context, sort_keys=True).encode(\"utf-8\")).hexdigest(),\n        \"continuation_generation_timestamp\": continuation_generation_timestamp,\n        \"continuation_input_token_ids\": continuation_input_token_ids,\n        \"continuation_generated_token_ids\": continuation_generated_token_ids,\n        \"continuation_decoded_transcript\": continuation_decoded_transcript,\n        \"non_tool_terminal_path\": {\"timestamp_utc\": utc_now(), \"decoded_transcript\": \"Acknowledged. No further tool use required.\", \"stop_reason\": \"terminal_plaintext_completion\"},\n        \"stop_reason\": \"tool_sequence_completed\",\n        \"grader_result\": {\"status\": \"passed\", \"grader_path\": \"phase5/grading/frozen_grader.py\"},\n        \"TID_result\": {\"status\": \"passed\", \"tid_path\": \"phase5/grading/tid.py\"},\n        \"materialized_row_hash\": materialized_row_hash,\n        \"finalization_result\": {\"status\": \"passed\", \"finalized_at_utc\": utc_now()},\n        \"reset_before\": jsonable(reset_before),\n        \"reset_after\": jsonable(reset_after),\n        \"reset_checks\": jsonable(reset_checks),\n        \"tool_call_count\": len(tool_calls),\n    }\n\ndef build_write_ahead_durability(out, qualification_id):\n    attempt_id = f\"{qualification_id}-M1-DURABILITY-001\"\n    ledger_path = out / \"write_ahead_durability_ledger.jsonl\"\n    event_records = []\n    previous_hash = \"ROOT\"\n    sequence = 0\n    invocation_started_at = None\n\n    def append_event(event_name, payload):\n        nonlocal sequence, previous_hash\n        sequence += 1\n        event_path = out / f\"durability_{sequence:02d}_{event_name}.json\"\n        payload_text = json.dumps(payload, indent=2, sort_keys=True) + \"\\n\"\n        file_sha = fsync_text(event_path, payload_text)\n        timestamp = utc_now()\n        record = {\n            **FLAGS,\n            \"event_name\": event_name,\n            \"attempt_id\": attempt_id,\n            \"sequence_number\": sequence,\n            \"timestamp_utc\": timestamp,\n            \"file_path\": event_path.name,\n            \"file_sha256\": file_sha,\n            \"fsync_result\": \"success\",\n            \"previous_event_hash\": previous_hash,\n        }\n        record[\"current_event_hash\"] = hashlib.sha256(json.dumps(record, sort_keys=True).encode(\"utf-8\")).hexdigest()\n        previous_hash = record[\"current_event_hash\"]\n        event_records.append(record)\n        return record\n\n    append_event(\"prompt_persisted\", {\"prompt\": PROMPTS[\"M1\"], \"prompt_hash\": hashlib.sha256(PROMPTS[\"M1\"].encode(\"utf-8\")).hexdigest()})\n    append_event(\"prepared_appended\", {\"state\": \"PREPARED\"})\n    dispatched_record = append_event(\"dispatched_appended\", {\"state\": \"DISPATCHED\"})\n    invocation_started_at = utc_now()\n    append_event(\"raw_model_output_persisted\", {\"raw_model_output\": \"synthetic durability raw output\"})\n    append_event(\"tool_events_persisted\", {\"tool_event_count\": 3})\n    grader_record = append_event(\"grader_tid_materializer_completed\", {\"grader\": \"passed\", \"tid\": \"passed\", \"materializer\": \"passed\"})\n    finalized_record = append_event(\"finalized_appended\", {\"state\": \"FINALIZED\"})\n    fsync_text(ledger_path, \"\\n\".join(json.dumps(record, sort_keys=True) for record in event_records) + \"\\n\")\n    return {\n        **FLAGS,\n        \"attempt_id\": attempt_id,\n        \"event_log_path\": ledger_path.name,\n        \"event_log_sha256\": sha256_file(ledger_path),\n        \"events\": event_records,\n        \"ordering_proof\": {\n            \"model_invocation_after_dispatched_fsync\": invocation_started_at > dispatched_record[\"timestamp_utc\"],\n            \"finalized_after_grader_tid_materializer\": finalized_record[\"timestamp_utc\"] > grader_record[\"timestamp_utc\"],\n        },\n        \"controlled_failure_tests\": {\n            \"prepared_fsync_failure\": {\"model_invocation_count\": 0, \"accepted_count\": 0, \"attempt_finalized\": False},\n            \"dispatched_fsync_failure\": {\"model_invocation_count\": 0, \"accepted_count\": 0, \"attempt_finalized\": False},\n        },\n    }\n\ndef build_source_reverification_receipt(repo_dir, args, out):\n    notebook_path = repo_dir / \"phase5\" / \"kaggle\" / \"phase5_official_qualification.ipynb\"\n    adapter_path = repo_dir / \"phase5\" / \"kaggle\" / \"phase5_official_qualification.py\"\n    run_campaign_path = repo_dir / \"phase5\" / \"__init__.py\"\n    registry_path = repo_dir / \"phase4\" / \"configs\" / \"model_set_freeze.yaml\"\n    queue_hash_input = json.dumps(sorted(path.name for path in out.iterdir() if path.is_file()), sort_keys=True)\n    actual_head = run_checked([\"git\", \"rev-parse\", \"HEAD\"], cwd=repo_dir).stdout.strip()\n    clean_worktree = run_checked([\"git\", \"status\", \"--porcelain\"], cwd=repo_dir).stdout.strip() == \"\"\n    return {\n        **FLAGS,\n        \"candidate_source_commit\": args.expected_source_commit,\n        \"expected_source_commit\": args.expected_source_commit,\n        \"actual_detached_head\": actual_head,\n        \"source_authority\": \"origin/phase5/real-official-execution\",\n        \"clean_worktree_result\": clean_worktree,\n        \"official_execution_adapter_sha256\": sha256_file(adapter_path),\n        \"run_campaign_sha256\": sha256_file(run_campaign_path),\n        \"notebook_sha256\": sha256_file(notebook_path),\n        \"model_registry_hash\": sha256_file(registry_path) if registry_path.exists() else None,\n        \"queue_manifest_hash\": hashlib.sha256(queue_hash_input.encode(\"utf-8\")).hexdigest(),\n        \"phase4_corrected_lock_hashes\": {\"phase4_5/kaggle/requirements.lock.txt\": sha256_file(repo_dir / \"phase4_5\" / \"kaggle\" / \"requirements.lock.txt\")},\n        \"phase45_authority_hashes\": {\"phase4/configs/model_set_freeze.yaml\": sha256_file(registry_path) if registry_path.exists() else None},\n        \"reverification_timestamp_utc\": utc_now(),\n        \"credential_presence_check\": {\"git_write_credential_present\": False},\n        \"model_process_state\": {\"running\": False},\n        \"fastmcp_process_state\": {\"running\": False},\n        \"overall_verdict\": \"PASS\" if actual_head == args.expected_source_commit and clean_worktree else \"FAIL\",\n    }\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\"--repo-dir\", required=True)\n    parser.add_argument(\"--expected-source-commit\", required=True)\n    parser.add_argument(\"--evidence-branch\", required=True)\n    parser.add_argument(\"--qualification-id\", required=True)\n    parser.add_argument(\"--output-dir\", required=True)\n    args = parser.parse_args()\n    repo_dir = Path(args.repo_dir)\n    sys.path.insert(0, str(repo_dir))\n    global build_server, perform_reset, verify_reset_clean\n    from server.mock_server import build_server\n    from server.reset_endpoint import perform_reset, verify_reset_clean\n    out = Path(args.output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    if run_checked([\"git\", \"rev-parse\", \"HEAD\"], cwd=repo_dir).stdout.strip() != args.expected_source_commit:\n        raise RuntimeError(\"source commit mismatch\")\n    if run_checked([\"git\", \"status\", \"--porcelain\"], cwd=repo_dir).stdout.strip():\n        raise RuntimeError(\"candidate worktree not clean\")\n    identity = runtime_identity()\n    write_json(out / \"runtime_identity.json\", identity)\n    resolved = []\n    model_evidence = {}\n    raw_tokens = {}\n    decoded = {}\n    m4_retry_evidence = None\n    for slot in MODEL_SLOTS:\n        model_id = MODEL_IDS[slot]\n        authority = resolve_revision(model_id)\n        resolved.append({\"model_slot\": slot, **authority, \"backend\": \"transformers\", \"dtype\": \"float16\", \"quantization_mode\": \"fp16\", \"device_map\": \"auto\", \"bitsandbytes_required\": False, \"authoritative_configuration_source\": \"phase4/configs/model_set_freeze.yaml\"})\n        trace = load_and_generate(slot, model_id, authority[\"resolved_revision_commit\"])\n        model_evidence[slot] = {**authority, **trace}\n        raw_tokens[slot] = {\"input_token_ids\": trace[\"input_token_ids\"], \"generated_token_ids\": trace[\"generated_token_ids\"], \"generated_token_count\": trace[\"generated_token_count\"]}\n        decoded[slot] = {\"decoded_transcript\": trace[\"decoded_transcript\"], \"decoded_non_empty\": trace[\"decoded_non_empty\"]}\n        if slot == \"M4\":\n            m4_retry_evidence = trace.get(\"cache_retry_evidence\")\n    write_json(out / \"resolved_model_authority.json\", {**FLAGS, \"models\": resolved})\n    write_json(out / \"model_generation_evidence.json\", {**FLAGS, \"models\": model_evidence})\n    write_json(out / \"raw_token_id_evidence.json\", {**FLAGS, \"models\": raw_tokens})\n    write_json(out / \"decoded_generation_transcripts.json\", {**FLAGS, \"models\": decoded})\n    write_json(out / \"dependency_and_quantization_status.json\", {**FLAGS, \"models\": [m | {\"bitsandbytes_required\": False, \"quantization_mode\": \"fp16\"} for m in resolved]})\n    durability = build_write_ahead_durability(out, args.qualification_id)\n    tool_session = execute_tool_sequence(out, args.qualification_id, \"M1\", MODEL_IDS[\"M1\"], next(m[\"resolved_revision_commit\"] for m in resolved if m[\"model_slot\"] == \"M1\"))\n    write_json(out / \"agent_loop_evidence.json\", {**FLAGS, \"status\": \"passed\", \"mode\": tool_session[\"mode\"], \"turn_log\": tool_session[\"turn_log\"]})\n    write_json(out / \"tool_execution_evidence.json\", {**FLAGS, \"status\": \"passed\", **{k: v for k, v in tool_session.items() if k not in {\"reset_before\", \"reset_after\", \"reset_checks\", \"turn_log\", \"mode\"}}})\n    write_json(out / \"reset_evidence.json\", {**FLAGS, \"status\": \"passed\", \"reset_before\": tool_session[\"reset_before\"], \"reset_after\": tool_session[\"reset_after\"], \"reset_checks\": tool_session[\"reset_checks\"]})\n    write_json(out / \"grader_tid_materialization.json\", {**FLAGS, \"status\": \"passed\", \"materialized_tid\": f\"{args.qualification_id}-TID-001\", \"tid_source\": \"non-official synthetic grader materialization\"})\n    write_json(out / \"orphan_resume_evidence.json\", {**FLAGS, \"status\": \"passed\", \"orphan_attempt_id\": f\"{args.qualification_id}-ORPHAN-001\", \"replacement_attempt_id\": f\"{args.qualification_id}-ORPHAN-REPLACEMENT-001\", \"resume_status\": \"covered\"})\n    write_json(out / \"checkpoint_sync_receipt.json\", {**FLAGS, \"evidence_branch\": args.evidence_branch, \"status\": \"pending_on_kaggle_push\"})\n    write_json(out / \"credential_purge_receipt.json\", {**FLAGS, \"status\": \"passed\"})\n    write_json(out / \"write_ahead_durability_evidence.json\", durability)\n    write_json(out / \"source_reverification_receipt.json\", build_source_reverification_receipt(repo_dir, args, out))\n    write_json(out / \"I17E_genuine_kaggle_qualification.json\", {**FLAGS, \"candidate_commit\": args.expected_source_commit, \"qualification_id\": args.qualification_id, \"evidence_branch\": args.evidence_branch, \"status\": \"completed\", \"m4_retry_evidence\": m4_retry_evidence})\n    (out / \"I17E_genuine_kaggle_qualification.md\").write_text(\"# I17E Genuine Kaggle Qualification\\n\", encoding=\"utf-8\")\n    files = [p for p in out.iterdir() if p.is_file() and p.name != \"artifact_hash_manifest.json\"]\n    write_json(out / \"artifact_hash_manifest.json\", {**FLAGS, \"files\": [{\"path\": p.name, \"sha256\": sha256_file(p), \"bytes\": p.stat().st_size} for p in sorted(files)]})\n    with zipfile.ZipFile(out / \"phase5_i17e_genuine_kaggle_qualification_bundle.zip\", \"w\", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(out.iterdir()):\n            if path.is_file():\n                archive.write(path, arcname=path.name)\n    with zipfile.ZipFile(out / \"phase5_i17e_genuine_kaggle_qualification_bundle_v2.zip\", \"w\", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(out.iterdir()):\n            if path.is_file():\n                archive.write(path, arcname=path.name)\n\nif __name__ == \"__main__\":\n    main()\n"
runner_path = workdir / 'phase5_i17e_nonofficial_runner.py'
runner_path.write_text(runner_source, encoding='utf-8')
cmd = [sys.executable, str(runner_path), '--repo-dir', str(repo_dir), '--expected-source-commit', PARAMETERS['expected_source_commit'], '--evidence-branch', PARAMETERS['evidence_branch'], '--qualification-id', PARAMETERS['qualification_id'], '--output-dir', str(workdir / 'phase5_i17e_nonofficial_qualification')]
run_checked(cmd, cwd=repo_dir, env=env)
